In [5]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Custom request headers to avoid being blocked by basic bot protections
REQUEST_HEADERS = {
    "User-Agent": "MyWebScraper/1.0"
}


def read_url_list(file_path: str) -> list:
    """
    Read a text file line by line and collect all valid URLs.
    Only lines starting with 'http' are treated as URLs.
    """
    urls = []

    # If file does not exist, just return an empty list
    if not os.path.isfile(file_path):
        return urls

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("http"):
                urls.append(line)

    return urls


def fetch_soup(url: str) -> BeautifulSoup | None:
    """
    Send a GET request to the given URL and return a BeautifulSoup object.
    Returns None if the request fails for any reason.。
    """
    try:
        response = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
        response.raise_for_status()  # Raise error for HTTP codes like 4xx/5xx
        return BeautifulSoup(response.text, "html.parser")
    except Exception:
        # In a real project you might log the error instead of silent pass
        return None


def extract_country_and_rank(soup: BeautifulSoup) -> pd.DataFrame:
    """
    From the main countries listing page, extract country names and overall rank.
    """
    country_cards = soup.select("div.picTrans.recordsetContainer")

    countries = []
    ranks = []

    for card in country_cards:
        try:
            # Country name
            name_el = card.find("span", class_="textWhite textLarge textShadow")
            # Global rank value
            rank_el = card.find("span", class_="textWhite textLarge textBold")

            country_name = name_el.text.strip()
            rank_value = rank_el.text.strip()

            countries.append(country_name)
            ranks.append(rank_value)
        except Exception:
            # Skip blocks that don't match the expected structure
            continue

    return pd.DataFrame(
        {
            "Country": countries,
            "Rank": ranks,
        }
    )


def extract_metric_from_page(url: str) -> pd.DataFrame | None:
    """
    For a specific Global Firepower metric page:
    - Parse all country entries
    - Extract the metric value for each country
    - Return a DataFrame with 'Country' and one metric column
    """
    soup = fetch_soup(url)
    if soup is None:
        return None

    metric_cards = soup.select("div.picTrans.recordsetContainer")
    if not metric_cards:
        return None

    countries = []
    values = []

    for card in metric_cards:
        try:
            # Country name
            name_el = card.find("span", class_="textWhite textLarge textShadow")
            # Metric value is usually in the last 'textWhite textLarge' span
            value_el = card.find_all("span", class_="textWhite textLarge")[-1]

            country_name = name_el.text.strip()
            metric_value = value_el.text.strip()

            countries.append(country_name)
            values.append(metric_value)
        except Exception:
            # Ignore entries that do not have the expected structure
            continue

    # Generate a usable metric column name from the URL
    metric_name = (
        url.split("/")[-1]  # get last part of URL
        .replace(".php", "")
        .replace("-", "_")
    )

    return pd.DataFrame(
        {
            "Country": countries,
            metric_name: values,
        }
    )


def clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    For all columns except 'Country' and 'Rank', try to:
    - Remove commas
    - Extract the numeric part
    Resulting values are left as strings (you can cast to float if needed).
    """
    for col in df.columns[2:]:
        df[col] = (
            df[col]
            .astype(str)  # Ensure string for regex operations
            .str.replace(",", "", regex=False)  # Remove thousands separators
            .str.extract(r"(\d+(?:\.\d+)?)")[0]  # Capture numeric pattern
        )
    return df


def build_global_firepower_dataset() -> pd.DataFrame:
    """
    Main function to:
    - Load list of metric URLs from file
    - Scrape base ranking page
    - Scrape each metric page and merge into a single DataFrame
    - Clean numeric columns
    """
    base_url = "https://www.globalfirepower.com/countries-listing.php"
    metric_urls = read_url_list("links_for_military_data.txt")
    print("Metric URLs found:", len(metric_urls), metric_urls) #debug

    # 1. Get base page data (country + rank)
    base_soup = fetch_soup(base_url)
    if base_soup is None:
        raise RuntimeError("Failed to load base countries listing page.")

    master_df = extract_country_and_rank(base_soup)
    print("Base columns:", master_df.columns.tolist())

    # 2. Loop through each metric page and merge its data
    for url in metric_urls:
      print("Scraping:",url)
      metric_df = extract_metric_from_page(url)
      if metric_df is None or metric_df.empty:
        print(" -> no data extracted from this page")
        continue
      print(" -> columns extracted:", metric_df.columns.tolist())

      # Merge on 'Country' so that each metric becomes a new column
      master_df = master_df.merge(metric_df, on="Country", how="left")

    # 3. Clean numeric-looking columns
    master_df = clean_numeric_columns(master_df)
    print("final columns:", master_df.columns.tolist())

    return master_df


if __name__ == "__main__":
    # Build the final dataset by scraping all pages
    final_df = build_global_firepower_dataset()

    # Save raw scraped data to CSV
    output_file = "military_raw_data.csv"
    final_df.to_csv(output_file, index=False)

    print(f"{output_file} created successfully!")

Metric URLs found: 54 ['https://www.globalfirepower.com/total-population-by-country.php', 'https://www.globalfirepower.com/available-military-manpower.php', 'https://www.globalfirepower.com/manpower-fit-for-military-service.php', 'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php', 'https://www.globalfirepower.com/active-military-manpower.php', 'https://www.globalfirepower.com/active-reserve-military-manpower.php', 'https://www.globalfirepower.com/manpower-paramilitary.php', 'https://www.globalfirepower.com/capital-cities-by-total-population.php', 'https://www.globalfirepower.com/aircraft-total.php', 'https://www.globalfirepower.com/aircraft-total-fighters.php', 'https://www.globalfirepower.com/aircraft-total-attack-types.php', 'https://www.globalfirepower.com/aircraft-total-transports.php', 'https://www.globalfirepower.com/aircraft-total-trainers.php', 'https://www.globalfirepower.com/aircraft-total-special-mission.php', 'https://www.globalfirepower.com/aircr